In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.ensemble import RandomForestRegressor

In [3]:
# Fetching stock data
stock_data = yf.download('RELIANCE.NS', start='2021-01-01', end='2022-12-31')

# Extracting all columns
data = pd.DataFrame(stock_data)
data

[*********************100%%**********************]  1 of 1 completed


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2021-01-01,1988.000000,1997.000000,1982.000000,1987.500000,1968.227539,4622002
2021-01-04,1995.099976,1998.900024,1968.000000,1990.849976,1971.545044,11312992
2021-01-05,1969.000000,1983.599976,1956.000000,1966.099976,1947.035034,11132803
2021-01-06,1965.900024,1966.000000,1905.150024,1914.250000,1895.687866,21414270
2021-01-07,1920.500000,1945.000000,1905.150024,1911.150024,1892.617920,14918406
...,...,...,...,...,...,...
2022-12-26,2514.750000,2542.000000,2492.399902,2524.050049,2515.165283,2764496
2022-12-27,2530.000000,2548.800049,2515.250000,2544.699951,2535.742676,2659749
2022-12-28,2538.000000,2549.800049,2521.500000,2544.449951,2535.493408,3442509


In [4]:
# Feature engineering: Adding a new column 'Next_Close' with the next day's closing price
data['Next_Close'] = data['Close'].shift(-1)

# Drop rows with NaN values
data = data.dropna()

In [5]:
# Feature engineering: Adding new column 'Next_Close' with next day's closing price
data['Next_Close'] = data['Close'].shift(-1)

# Drop rows with NaN values
data = data.dropna()

# Feature scaling
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)
data

C:\Users\bvsat\AppData\Local\Temp\ipykernel_2504\1805377570.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Next_Close'] = data['Close'].shift(-1)


,Open,High,Low,Close,Adj Close,Volume,Next_Close
Date,,,,,,,
2021-01-01,1988.000000,1997.000000,1982.000000,1987.500000,1968.227539,4622002,1990.849976
2021-01-04,1995.099976,1998.900024,1968.000000,1990.849976,1971.545044,11312992,1966.099976
2021-01-05,1969.000000,1983.599976,1956.000000,1966.099976,1947.035034,11132803,1914.250000
2021-01-06,1965.900024,1966.000000,1905.150024,1914.250000,1895.687866,21414270,1911.150024
2021-01-07,1920.500000,1945.000000,1905.150024,1911.150024,1892.617920,14918406,1933.699951
...,...,...,...,...,...,...,...
2022-12-22,2598.000000,2604.649902,2566.750000,2577.800049,2568.726074,3438692,2502.199951
2022-12-23,2563.300049,2590.500000,2492.250000,2502.199951,2493.392090,4733657,2524.050049
2022-12-26,2514.750000,2542.000000,2492.399902,2524.050049,2515.165283,2764496,2544.699951


In [6]:
# Feature scaling
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data[['Open', 'High', 'Low', 'Close', 'Volume']])

# Splitting the data into training and validation sets
train_size = int(len(data) * 0.8)
train_data, validation_data = data_scaled[:train_size], data_scaled[train_size:]

In [7]:
# Function to create sequences for LSTM
def create_sequences(data, seq_length):
    sequences, targets = [], []
    for i in range(len(data) - seq_length):
        seq = data[i:i + seq_length, :]
        target = data[i + seq_length, 3]  # Assuming 'Close' column is the target
        sequences.append(seq)
        targets.append(target)
    return np.array(sequences), np.array(targets)

In [8]:
# Creating sequences for LSTM
seq_length = 20
X_train, y_train = create_sequences(train_data, seq_length)
X_validation, y_validation = create_sequences(validation_data, seq_length)


In [9]:
# LSTM model
model = Sequential()
model.add(LSTM(10, input_shape=(seq_length, X_train.shape[2])))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=200, batch_size=32, verbose=2)



Epoch 1/200

12/12 - 2s - loss: 0.2344 - 2s/epoch - 161ms/step
Epoch 2/200
12/12 - 0s - loss: 0.0951 - 75ms/epoch - 6ms/step
Epoch 3/200
12/12 - 0s - loss: 0.0253 - 70ms/epoch - 6ms/step
Epoch 4/200
12/12 - 0s - loss: 0.0134 - 69ms/epoch - 6ms/step
Epoch 5/200
12/12 - 0s - loss: 0.0117 - 69ms/epoch - 6ms/step
Epoch 6/200
12/12 - 0s - loss: 0.0093 - 68ms/epoch - 6ms/step
Epoch 7/200
12/12 - 0s - loss: 0.0081 - 67ms/epoch - 6ms/step
Epoch 8/200
12/12 - 0s - loss: 0.0073 - 67ms/epoch - 6ms/step
Epoch 9/200
12/12 - 0s - loss: 0.0066 - 68ms/epoch - 6ms/step
Epoch 10/200
12/12 - 0s - loss: 0.0061 - 70ms/epoch - 6ms/step
Epoch 11/200
12/12 - 0s - loss: 0.0057 - 71ms/epoch - 6ms/step
Epoch 12/200
12/12 - 0s - loss: 0.0054 - 69ms/epoch - 6ms/step
Epoch 13/200
12/12 - 0s - loss: 0.0052 - 70ms/epoch - 6ms/step
Epoch 14/200
12/12 - 0s - loss: 0.0051 - 69ms/epoch - 6ms/step
Epoch 15/200
12/12 - 0s - loss: 0.0049 - 76ms/epoch - 6ms/step
Epoch 16/200
12/12 - 0s - loss: 0.0048 - 74ms/epoch - 6ms/ste

Epoch 122/200
12/12 - 0s - loss: 0.0025 - 70ms/epoch - 6ms/step
Epoch 123/200
12/12 - 0s - loss: 0.0025 - 71ms/epoch - 6ms/step
Epoch 124/200
12/12 - 0s - loss: 0.0025 - 71ms/epoch - 6ms/step
Epoch 125/200
12/12 - 0s - loss: 0.0025 - 68ms/epoch - 6ms/step
Epoch 126/200
12/12 - 0s - loss: 0.0024 - 69ms/epoch - 6ms/step
Epoch 127/200
12/12 - 0s - loss: 0.0024 - 69ms/epoch - 6ms/step
Epoch 128/200
12/12 - 0s - loss: 0.0024 - 66ms/epoch - 5ms/step
Epoch 129/200
12/12 - 0s - loss: 0.0024 - 69ms/epoch - 6ms/step
Epoch 130/200
12/12 - 0s - loss: 0.0024 - 68ms/epoch - 6ms/step
Epoch 131/200
12/12 - 0s - loss: 0.0024 - 66ms/epoch - 5ms/step
Epoch 132/200
12/12 - 0s - loss: 0.0025 - 68ms/epoch - 6ms/step
Epoch 133/200
12/12 - 0s - loss: 0.0025 - 67ms/epoch - 6ms/step
Epoch 134/200
12/12 - 0s - loss: 0.0024 - 71ms/epoch - 6ms/step
Epoch 135/200
12/12 - 0s - loss: 0.0024 - 68ms/epoch - 6ms/step
Epoch 136/200
12/12 - 0s - loss: 0.0024 - 69ms/epoch - 6ms/step
Epoch 137/200
12/12 - 0s - loss: 0.0024 

In [10]:
# Making predictions with LSTM on validation data
lstm_predictions_validation = model.predict(X_validation)
lstm_predictions_validation = lstm_predictions_validation.reshape(-1)

3/3 [==============================] - 0s 3ms/step


In [11]:
# Decision Tree model
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train[:, -1, :], y_train)
dt_predictions_validation = dt_model.predict(X_validation[:, -1, :])

In [12]:
# Random Forest model
rf_model = RandomForestRegressor(n_estimators=10, random_state=42)
rf_model.fit(X_train[:, -1, :], y_train)

# Making predictions with Random Forest on validation data
rf_predictions_validation = rf_model.predict(X_validation[:, -1, :])

In [13]:
# ARIMA model
arima_model = ARIMA(train_data[:, 3], order=(5, 1, 0))  # Assuming 'Close' column is the target
arima_model_fit = arima_model.fit()
arima_predictions_validation = arima_model_fit.forecast(steps=len(X_validation))

In [14]:
# SARIMA model
sarima_model = SARIMAX(train_data[:, 3], order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))  # Assuming 'Close' column is the target
sarima_model_fit = sarima_model.fit()
sarima_predictions_validation = sarima_model_fit.get_forecast(steps=len(X_validation)).predicted_mean


In [15]:
# Calculate R^2 and MSE for each model on the validation set
r2_lstm_validation = r2_score(y_validation, lstm_predictions_validation)
r2_dt_validation = r2_score(y_validation, dt_predictions_validation)
r2_rf_validation = r2_score(y_validation, rf_predictions_validation)
r2_arima_validation = r2_score(y_validation, arima_predictions_validation)
r2_sarima_validation = r2_score(y_validation, sarima_predictions_validation)

ValueError: Found input variables with inconsistent numbers of samples: [79, 158]

In [ ]:
mse_lstm_validation = mean_squared_error(y_validation, lstm_predictions_validation)
mse_dt_validation = mean_squared_error(y_validation, dt_predictions_validation)
mse_rf_validation = mean_squared_error(y_validation, rf_predictions_validation)
mse_arima_validation = mean_squared_error(y_validation, arima_predictions_validation)
mse_sarima_validation = mean_squared_error(y_validation, sarima_predictions_validation)

In [16]:
# Display R^2 and MSE for each model on the validation data
print("\nValidation Model Performance:")
print("{:<25} {:<25} {:<25}".format("Model", "R^2 Score (Accuracy)", "MSE"))
print("-" * 65)
print("{:<25} {:<25} {:<25}".format("LSTM", r2_lstm_validation, mse_lstm_validation))
print("{:<25} {:<25} {:<25}".format("Decision Tree", r2_dt_validation, mse_dt_validation))
#print("{:<25} {:<25} {:<25}".format("Bagging", r2_bagging_validation, mse_bagging_validation))
print("{:<25} {:<25} {:<25}".format("Random Forest", r2_rf_validation, mse_rf_validation))
print("{:<25} {:<25} {:<25}".format("ARIMA", r2_arima_validation, mse_arima_validation))
print("{:<25} {:<25} {:<25}".format("SARIMA", r2_sarima_validation, mse_sarima_validation))


Validation Model Performance:
Model                     R^2 Score (Accuracy)      MSE                      
-----------------------------------------------------------------


NameError: name 'r2_lstm_validation' is not defined

In [144]:
# Calculate Mean Squared Error (MSE) for each model
mse_lstm = mean_squared_error(data['Next_Close'][train_size + seq_length:], lstm_predictions_validation)
mse_dt = mean_squared_error(data['Next_Close'][train_size + seq_length:], dt_predictions_validation)
mse_rf = mean_squared_error(data['Next_Close'][train_size + seq_length:], rf_predictions_validation)
mse_arima = mean_squared_error(data['Next_Close'][train_size + seq_length:], arima_predictions_validation)
mse_sarima = mean_squared_error(data['Next_Close'][train_size + seq_length:], sarima_predictions_validation)

In [148]:
Actual_unscaled = data['Next_Close'][train_size + seq_length:]

In [149]:
Actual_unscaled

Date
2022-09-06    2581.750000
2022-09-07    2585.399902
2022-09-08    2569.300049
2022-09-09    2598.050049
2022-09-12    2619.750000
                 ...     
2022-12-22    2502.199951
2022-12-23    2524.050049
2022-12-26    2544.699951
2022-12-27    2544.449951
2022-12-28    2543.300049
Name: Next_Close, Length: 79, dtype: float64

In [150]:
# Feature scaling
scaler = MinMaxScaler()
x = scaler.fit_transform(Actual_unscaled.values.reshape(-1, 1))
x

array([[0.63157227],
       [0.64056107],
       [0.60091115],
       [0.67171523],
       [0.72515679],
       [0.64782666],
       [0.58465675],
       [0.42827209],
       [0.43627603],
       [0.43726149],
       [0.45425423],
       [0.39601042],
       [0.281246  ],
       [0.12818629],
       [0.17473203],
       [0.01760843],
       [0.        ],
       [0.12917115],
       [0.10885346],
       [0.21647554],
       [0.23839438],
       [0.26363757],
       [0.19825142],
       [0.08028531],
       [0.12301427],
       [0.14160816],
       [0.11180864],
       [0.20551642],
       [0.31018332],
       [0.41521939],
       [0.4317203 ],
       [0.36030053],
       [0.38061822],
       [0.28629476],
       [0.3099368 ],
       [0.4946431 ],
       [0.55239508],
       [0.50375575],
       [0.54266673],
       [0.56606286],
       [0.65866253],
       [0.69277186],
       [0.68636847],
       [0.60879183],
       [0.75483306],
       [0.72343299],
       [0.69449567],
       [0.657

In [153]:
# Creating a DataFrame with actual and predicted prices
actual_values_scaled = scaler.transform(data['Next_Close'].iloc[train_size + seq_length:].values.reshape(-1, 1))
predictions_df = pd.DataFrame({
    'Actual': actual_values_scaled.flatten(),
    'LSTM_Predicted': lstm_predictions_validation,
    'RandomForest_Predicted': rf_predictions_validation,
    'Arima_Predicted': arima_predictions_validation,
    'Sarima_Predicted': sarima_predictions_validation,

})
predictions_df

,Actual,LSTM_Predicted,RandomForest_Predicted,Arima_Predicted,Sarima_Predicted
0,0.631572,0.728094,0.774957,0.776347,0.771238
1,0.640561,0.756461,0.741553,0.775550,0.772122
2,0.600911,0.746403,0.757286,0.773193,0.780948
3,0.671715,0.750060,0.760758,0.773504,0.793955
4,0.725157,0.744717,0.717425,0.773540,0.790655
...,...,...,...,...,...
74,0.435660,0.762682,0.738808,0.773749,0.892121
75,0.489472,0.749418,0.744790,0.773749,0.905191
76,0.540327,0.692460,0.648231,0.773749,0.900524
77,0.539712,0.692517,0.714562,0.773749,0.899879
